## Objectives

XGBoost + Signature
XGBoost + GARCH
XGBoost + Combined

CNN + Signature
CNN + GARCH
CNN + Combined

CNN/XGBoost Ensemble

In [56]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [57]:
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [58]:
FEATURE_DIR = Path("../data/features/monthly_ff5")
MODEL_DIR = Path("../models/monthly_ff5")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [59]:
X_signature_train = np.load(
    FEATURE_DIR / "X_signature_train.npy"
)

X_signature_validation = np.load(
    FEATURE_DIR / "X_signature_validation.npy"
)

X_signature_test = np.load(
    FEATURE_DIR / "X_signature_test.npy"
)

X_garch_train = np.load(
    FEATURE_DIR / "X_garch_train.npy"
)

X_garch_validation = np.load(
    FEATURE_DIR / "X_garch_validation.npy"
)

X_garch_test = np.load(
    FEATURE_DIR / "X_garch_test.npy"
)

X_combined_train = np.load(
    FEATURE_DIR / "X_combined_train.npy"
)

X_combined_validation = np.load(
    FEATURE_DIR / "X_combined_validation.npy"
)

X_combined_test = np.load(
    FEATURE_DIR / "X_combined_test.npy"
)

y_train = np.load(FEATURE_DIR / "y_train.npy")
y_validation = np.load(FEATURE_DIR / "y_validation.npy")
y_test = np.load(FEATURE_DIR / "y_test.npy")

In [60]:
feature_sets = {
    "signature": (
        X_signature_train,
        X_signature_validation,
        X_signature_test
    ),
    "garch": (
        X_garch_train,
        X_garch_validation,
        X_garch_test
    ),
    "combined": (
        X_combined_train,
        X_combined_validation,
        X_combined_test
    )
}

for name, (X_tr, X_val, X_te) in feature_sets.items():
    print(name)
    print("Train:", X_tr.shape, y_train.shape)
    print("Validation:", X_val.shape, y_validation.shape)
    print("Test:", X_te.shape, y_test.shape)
    print()

signature
Train: (12950, 36) (12950,)
Validation: (2775, 36) (2775,)
Test: (2800, 36) (2800,)

garch
Train: (12950, 12) (12950,)
Validation: (2775, 12) (2775,)
Test: (2800, 12) (2800,)

combined
Train: (12950, 48) (12950,)
Validation: (2775, 48) (2775,)
Test: (2800, 48) (2800,)



In [61]:
def show_class_balance(y, split_name):
    values, counts = np.unique(y, return_counts=True)

    print(split_name)

    for value, count in zip(values, counts):
        print(
            f"Class {value}: "
            f"{count} observations "
            f"({count / len(y):.2%})"
        )

    print()

In [62]:
show_class_balance(y_train, "Training")
show_class_balance(y_validation, "Validation")
show_class_balance(y_test, "Testing")

Training
Class 0: 10360 observations (80.00%)
Class 1: 2590 observations (20.00%)

Validation
Class 0: 2220 observations (80.00%)
Class 1: 555 observations (20.00%)

Testing
Class 0: 2240 observations (80.00%)
Class 1: 560 observations (20.00%)



In [63]:
def evaluate_classifier(
    model_name,
    y_true,
    probabilities,
    threshold=0.50
):
    """
    Evaluate binary-classification probabilities at a chosen threshold.
    """

    predictions = (probabilities >= threshold).astype(int)

    results = {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        )
    }

    print("=" * 70)
    print(model_name)
    print("=" * 70)

    for metric, value in results.items():
        if metric != "model":
            print(f"{metric}: {value:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            predictions,
            zero_division=0
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(y_true, predictions))

    return results

In [64]:
negative_count = np.sum(y_train == 0)
positive_count = np.sum(y_train == 1)

scale_pos_weight = negative_count / positive_count

print("Negative observations:", negative_count)
print("Positive observations:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

Negative observations: 10360
Positive observations: 2590
scale_pos_weight: 4.0


In [65]:
def train_xgboost_model(
    X_train,
    y_train,
    X_validation,
    y_validation,
    scale_pos_weight,
    random_state=42
):
    """
    Train an XGBoost binary classifier.
    """

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",

        n_estimators=1000,
        learning_rate=0.03,
        max_depth=4,

        min_child_weight=3,
        gamma=0.0,

        subsample=0.80,
        colsample_bytree=0.80,

        reg_alpha=0.0,
        reg_lambda=1.0,

        scale_pos_weight=scale_pos_weight,

        random_state=random_state,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (X_train, y_train),
            (X_validation, y_validation)
        ],
        verbose=False
    )

    return model

In [66]:
xgb_signature = train_xgboost_model(
    X_train=X_signature_train,
    y_train=y_train,
    X_validation=X_signature_validation,
    y_validation=y_validation,
    scale_pos_weight=scale_pos_weight
)

In [67]:
signature_validation_prob = xgb_signature.predict_proba(
    X_signature_validation
)[:, 1]

signature_validation_result = evaluate_classifier(
    model_name="XGBoost Signature — Validation",
    y_true=y_validation,
    probabilities=signature_validation_prob
)

XGBoost Signature — Validation
threshold: 0.5000
accuracy: 0.6652
precision: 0.1984
recall: 0.2216
f1: 0.2094
roc_auc: 0.5060

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.78      0.79      2220
           1       0.20      0.22      0.21       555

    accuracy                           0.67      2775
   macro avg       0.50      0.50      0.50      2775
weighted avg       0.68      0.67      0.67      2775

Confusion matrix:
[[1723  497]
 [ 432  123]]


In [68]:
xgb_garch = train_xgboost_model(
    X_train=X_garch_train,
    y_train=y_train,
    X_validation=X_garch_validation,
    y_validation=y_validation,
    scale_pos_weight=scale_pos_weight
)

In [69]:
garch_validation_prob = xgb_garch.predict_proba(
    X_garch_validation
)[:, 1]

garch_validation_result = evaluate_classifier(
    model_name="XGBoost GARCH — Validation",
    y_true=y_validation,
    probabilities=garch_validation_prob
)

XGBoost GARCH — Validation
threshold: 0.5000
accuracy: 0.6119
precision: 0.1915
recall: 0.2919
f1: 0.2313
roc_auc: 0.5073

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.69      0.74      2220
           1       0.19      0.29      0.23       555

    accuracy                           0.61      2775
   macro avg       0.49      0.49      0.49      2775
weighted avg       0.68      0.61      0.64      2775

Confusion matrix:
[[1536  684]
 [ 393  162]]


In [70]:
xgb_combined = train_xgboost_model(
    X_train=X_combined_train,
    y_train=y_train,
    X_validation=X_combined_validation,
    y_validation=y_validation,
    scale_pos_weight=scale_pos_weight
)

In [71]:
combined_validation_prob = xgb_combined.predict_proba(
    X_combined_validation
)[:, 1]

combined_validation_result = evaluate_classifier(
    model_name="XGBoost Combined — Validation",
    y_true=y_validation,
    probabilities=combined_validation_prob
)

XGBoost Combined — Validation
threshold: 0.5000
accuracy: 0.6879
precision: 0.2208
recall: 0.2216
f1: 0.2212
roc_auc: 0.5136

Classification report:
              precision    recall  f1-score   support

           0       0.81      0.80      0.80      2220
           1       0.22      0.22      0.22       555

    accuracy                           0.69      2775
   macro avg       0.51      0.51      0.51      2775
weighted avg       0.69      0.69      0.69      2775

Confusion matrix:
[[1786  434]
 [ 432  123]]


In [72]:
xgb_validation_results = pd.DataFrame([
    signature_validation_result,
    garch_validation_result,
    combined_validation_result
])

xgb_validation_results = xgb_validation_results.sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)

display(xgb_validation_results)

,model,threshold,accuracy,precision,recall,f1,roc_auc
0,XGBoost GARCH — Validation,0.5,0.611892,0.191489,0.291892,0.231263,0.507337
1,XGBoost Combined — Validation,0.5,0.687928,0.220826,0.221622,0.221223,0.513591
2,XGBoost Signature — Validation,0.5,0.665225,0.198387,0.221622,0.209362,0.506038


In [73]:
def find_best_threshold(
    y_true,
    probabilities,
    metric="f1"
):
    """
    Find the probability threshold that maximizes F1.
    """

    thresholds = np.arange(0.05, 0.96, 0.01)
    records = []

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        record = {
            "threshold": threshold,
            "precision": precision_score(
                y_true,
                predictions,
                zero_division=0
            ),
            "recall": recall_score(
                y_true,
                predictions,
                zero_division=0
            ),
            "f1": f1_score(
                y_true,
                predictions,
                zero_division=0
            )
        }

        records.append(record)

    threshold_results = pd.DataFrame(records)

    best_row = threshold_results.loc[
        threshold_results[metric].idxmax()
    ]

    return best_row, threshold_results

In [74]:
best_signature_threshold, signature_threshold_table = find_best_threshold(
    y_validation,
    signature_validation_prob
)

best_garch_threshold, garch_threshold_table = find_best_threshold(
    y_validation,
    garch_validation_prob
)

best_combined_threshold, combined_threshold_table = find_best_threshold(
    y_validation,
    combined_validation_prob
)

In [75]:
best_thresholds = pd.DataFrame({
    "feature_set": [
        "signature",
        "garch",
        "combined"
    ],
    "threshold": [
        best_signature_threshold["threshold"],
        best_garch_threshold["threshold"],
        best_combined_threshold["threshold"]
    ],
    "precision": [
        best_signature_threshold["precision"],
        best_garch_threshold["precision"],
        best_combined_threshold["precision"]
    ],
    "recall": [
        best_signature_threshold["recall"],
        best_garch_threshold["recall"],
        best_combined_threshold["recall"]
    ],
    "f1": [
        best_signature_threshold["f1"],
        best_garch_threshold["f1"],
        best_combined_threshold["f1"]
    ]
})

display(best_thresholds)

,feature_set,threshold,precision,recall,f1
0,signature,0.09,0.200072,1.000000,0.333433
1,garch,0.19,0.201628,0.981982,0.334561
2,combined,0.09,0.199856,0.998198,0.333033


In [76]:
signature_test_prob = xgb_signature.predict_proba(
    X_signature_test
)[:, 1]

garch_test_prob = xgb_garch.predict_proba(
    X_garch_test
)[:, 1]

combined_test_prob = xgb_combined.predict_proba(
    X_combined_test
)[:, 1]

In [77]:
signature_threshold = float(
    best_signature_threshold["threshold"]
)

garch_threshold = float(
    best_garch_threshold["threshold"]
)

combined_threshold = float(
    best_combined_threshold["threshold"]
)

In [78]:
xgb_signature_test_result = evaluate_classifier(
    model_name="XGBoost Signature — Test",
    y_true=y_test,
    probabilities=signature_test_prob,
    threshold=signature_threshold
)

xgb_garch_test_result = evaluate_classifier(
    model_name="XGBoost GARCH — Test",
    y_true=y_test,
    probabilities=garch_test_prob,
    threshold=garch_threshold
)

xgb_combined_test_result = evaluate_classifier(
    model_name="XGBoost Combined — Test",
    y_true=y_test,
    probabilities=combined_test_prob,
    threshold=combined_threshold
)

XGBoost Signature — Test
threshold: 0.0900
accuracy: 0.2104
precision: 0.1995
recall: 0.9786
f1: 0.3314
roc_auc: 0.5015

Classification report:
              precision    recall  f1-score   support

           0       0.77      0.02      0.04      2240
           1       0.20      0.98      0.33       560

    accuracy                           0.21      2800
   macro avg       0.49      0.50      0.18      2800
weighted avg       0.66      0.21      0.09      2800

Confusion matrix:
[[  41 2199]
 [  12  548]]
XGBoost GARCH — Test
threshold: 0.1900
accuracy: 0.2164
precision: 0.2010
recall: 0.9804
f1: 0.3335
roc_auc: 0.4982

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.03      0.05      2240
           1       0.20      0.98      0.33       560

    accuracy                           0.22      2800
   macro avg       0.52      0.50      0.19      2800
weighted avg       0.71      0.22      0.11      2800

Confusion matrix:

In [79]:
xgb_test_results = pd.DataFrame([
    xgb_signature_test_result,
    xgb_garch_test_result,
    xgb_combined_test_result
]).sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)

display(xgb_test_results)

,model,threshold,accuracy,precision,recall,f1,roc_auc
0,XGBoost GARCH — Test,0.19,0.216429,0.200952,0.980357,0.333536,0.498162
1,XGBoost Combined — Test,0.09,0.200714,0.199283,0.992857,0.331940,0.499207
2,XGBoost Signature — Test,0.09,0.210357,0.199490,0.978571,0.331418,0.501473


In [80]:
xgb_signature.save_model(
    MODEL_DIR / "xgb_signature.json"
)

xgb_garch.save_model(
    MODEL_DIR / "xgb_garch.json"
)

xgb_combined.save_model(
    MODEL_DIR / "xgb_combined.json"
)

In [81]:
thresholds_to_save = {
    "signature": signature_threshold,
    "garch": garch_threshold,
    "combined": combined_threshold
}

pd.Series(
    thresholds_to_save,
    name="threshold"
).to_csv(
    MODEL_DIR / "xgb_thresholds.csv"
)

xgb_validation_results.to_csv(
    MODEL_DIR / "xgb_validation_results.csv",
    index=False
)

xgb_test_results.to_csv(
    MODEL_DIR / "xgb_test_results.csv",
    index=False
)

## CNN Section

In [82]:
def reshape_for_cnn(X):
    """
    Convert 2D tabular features into 3D CNN input.
    """
    return X.reshape(
        X.shape[0],
        X.shape[1],
        1
    )

In [83]:
cnn_feature_sets = {
    "signature": (
        reshape_for_cnn(X_signature_train),
        reshape_for_cnn(X_signature_validation),
        reshape_for_cnn(X_signature_test)
    ),
    "garch": (
        reshape_for_cnn(X_garch_train),
        reshape_for_cnn(X_garch_validation),
        reshape_for_cnn(X_garch_test)
    ),
    "combined": (
        reshape_for_cnn(X_combined_train),
        reshape_for_cnn(X_combined_validation),
        reshape_for_cnn(X_combined_test)
    )
}

In [84]:
for name, (X_tr, X_val, X_te) in cnn_feature_sets.items():
    print(name)
    print("Train:", X_tr.shape)
    print("Validation:", X_val.shape)
    print("Test:", X_te.shape)
    print()

signature
Train: (12950, 36, 1)
Validation: (2775, 36, 1)
Test: (2800, 36, 1)

garch
Train: (12950, 12, 1)
Validation: (2775, 12, 1)
Test: (2800, 12, 1)

combined
Train: (12950, 48, 1)
Validation: (2775, 48, 1)
Test: (2800, 48, 1)



In [85]:
def build_cnn_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv1D(
            filters=32,
            kernel_size=3,
            padding="same",
            activation="relu"
        ),
        layers.BatchNormalization(),

        layers.Conv1D(
            filters=64,
            kernel_size=3,
            padding="same",
            activation="relu"
        ),
        layers.BatchNormalization(),

        layers.GlobalAveragePooling1D(),

        layers.Dense(
            64,
            activation="relu"
        ),
        layers.Dropout(0.30),

        layers.Dense(
            32,
            activation="relu"
        ),
        layers.Dropout(0.20),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.Precision(
                name="precision"
            ),
            tf.keras.metrics.Recall(
                name="recall"
            ),
            tf.keras.metrics.AUC(
                name="auc"
            )
        ]
    )

    return model

In [86]:
class_weight = {
    0: 1.0,
    1: scale_pos_weight
}

print(class_weight)

{0: 1.0, 1: np.float64(4.0)}


In [87]:
def train_cnn_model(
    X_train,
    y_train,
    X_validation,
    y_validation,
    model_name,
    epochs=100,
    batch_size=64
):
    model = build_cnn_model(
        input_shape=X_train.shape[1:]
    )

    checkpoint_path = (
        MODEL_DIR / f"{model_name}.keras"
    )

    callback_list = [
        callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=12,
            restore_best_weights=True
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6
        ),
        callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True
        )
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(
            X_validation,
            y_validation
        ),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight,
        callbacks=callback_list,
        verbose=1
    )

    return model, history

In [88]:
(
    X_signature_train_cnn,
    X_signature_validation_cnn,
    X_signature_test_cnn
) = cnn_feature_sets["signature"]

In [89]:
cnn_signature, history_signature = train_cnn_model(
    X_train=X_signature_train_cnn,
    y_train=y_train,
    X_validation=X_signature_validation_cnn,
    y_validation=y_validation,
    model_name="cnn_signature"
)

Epoch 1/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.4797 - auc: 0.4858 - loss: 1.1192 - precision: 0.1957 - recall: 0.5151 - val_accuracy: 0.8000 - val_auc: 0.5036 - val_loss: 0.6669 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.5213 - auc: 0.5032 - loss: 1.1120 - precision: 0.2010 - recall: 0.4683 - val_accuracy: 0.5820 - val_auc: 0.5009 - val_loss: 0.6861 - val_precision: 0.2077 - val_recall: 0.3874 - learning_rate: 0.0010
Epoch 3/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.5503 - auc: 0.5036 - loss: 1.1108 - precision: 0.2023 - recall: 0.4243 - val_accuracy: 0.6768 - val_auc: 0.5080 - val_loss: 0.6827 - val_precision: 0.2072 - val_recall: 0.2180 - learning_rate: 0.0010
Epoch 4/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.5240 - auc: 0.5012 - loss: 1.1096 - precision: 0.2004 - recall: 0.4614 - val_accuracy: 0.5452 - val_auc: 0.4970 - val_l

In [90]:
(
    X_garch_train_cnn,
    X_garch_validation_cnn,
    X_garch_test_cnn
) = cnn_feature_sets["garch"]

In [91]:
cnn_garch, history_garch = train_cnn_model(
    X_train=X_garch_train_cnn,
    y_train=y_train,
    X_validation=X_garch_validation_cnn,
    y_validation=y_validation,
    model_name="cnn_garch"
)

Epoch 1/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.5072 - auc: 0.5027 - loss: 1.1195 - precision: 0.1975 - recall: 0.4780 - val_accuracy: 0.7986 - val_auc: 0.5060 - val_loss: 0.6738 - val_precision: 0.3333 - val_recall: 0.0072 - learning_rate: 0.0010
Epoch 2/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.5297 - auc: 0.5057 - loss: 1.1134 - precision: 0.1989 - recall: 0.4463 - val_accuracy: 0.3780 - val_auc: 0.4804 - val_loss: 0.7075 - val_precision: 0.1914 - val_recall: 0.6541 - learning_rate: 0.0010
Epoch 3/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.4999 - auc: 0.5015 - loss: 1.1118 - precision: 0.2002 - recall: 0.5012 - val_accuracy: 0.4533 - val_auc: 0.4796 - val_loss: 0.6998 - val_precision: 0.1860 - val_recall: 0.5135 - learning_rate: 0.0010
Epoch 4/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.5147 - auc: 0.5124 - loss: 1.1103 - precision: 0.2032 - recall: 0.4880 - val_accuracy: 0.5499 - val_auc: 0.4919 - val_loss: 0.6

In [92]:
(
    X_combined_train_cnn,
    X_combined_validation_cnn,
    X_combined_test_cnn
) = cnn_feature_sets["combined"]

In [93]:
cnn_combined, history_combined = train_cnn_model(
    X_train=X_combined_train_cnn,
    y_train=y_train,
    X_validation=X_combined_validation_cnn,
    y_validation=y_validation,
    model_name="cnn_combined"
)

Epoch 1/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.4776 - auc: 0.4895 - loss: 1.1147 - precision: 0.1940 - recall: 0.5108 - val_accuracy: 0.2011 - val_auc: 0.4808 - val_loss: 0.7111 - val_precision: 0.2000 - val_recall: 0.9982 - learning_rate: 0.0010
Epoch 2/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.4884 - auc: 0.4981 - loss: 1.1111 - precision: 0.1983 - recall: 0.5120 - val_accuracy: 0.5802 - val_auc: 0.4839 - val_loss: 0.6918 - val_precision: 0.2016 - val_recall: 0.3712 - learning_rate: 0.0010
Epoch 3/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.5268 - auc: 0.4982 - loss: 1.1110 - precision: 0.1983 - recall: 0.4490 - val_accuracy: 0.5283 - val_auc: 0.4953 - val_loss: 0.6922 - val_precision: 0.2003 - val_recall: 0.4541 - learning_rate: 0.0010
Epoch 4/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.5208 - auc: 0.5058 - loss: 1.1103 - precision: 0.2003 - recall: 0.4664 - val_accuracy: 0.6919 - val_auc: 0.4918 - val_loss: 0.68

In [94]:
cnn_signature_validation_prob = cnn_signature.predict(
    X_signature_validation_cnn,
    verbose=0
).ravel()

cnn_garch_validation_prob = cnn_garch.predict(
    X_garch_validation_cnn,
    verbose=0
).ravel()

cnn_combined_validation_prob = cnn_combined.predict(
    X_combined_validation_cnn,
    verbose=0
).ravel()

In [95]:
best_cnn_signature_threshold, _ = find_best_threshold(
    y_validation,
    cnn_signature_validation_prob
)

best_cnn_garch_threshold, _ = find_best_threshold(
    y_validation,
    cnn_garch_validation_prob
)

best_cnn_combined_threshold, _ = find_best_threshold(
    y_validation,
    cnn_combined_validation_prob
)

In [96]:
cnn_thresholds = pd.DataFrame({
    "feature_set": [
        "signature",
        "garch",
        "combined"
    ],
    "threshold": [
        best_cnn_signature_threshold["threshold"],
        best_cnn_garch_threshold["threshold"],
        best_cnn_combined_threshold["threshold"]
    ],
    "precision": [
        best_cnn_signature_threshold["precision"],
        best_cnn_garch_threshold["precision"],
        best_cnn_combined_threshold["precision"]
    ],
    "recall": [
        best_cnn_signature_threshold["recall"],
        best_cnn_garch_threshold["recall"],
        best_cnn_combined_threshold["recall"]
    ],
    "f1": [
        best_cnn_signature_threshold["f1"],
        best_cnn_garch_threshold["f1"],
        best_cnn_combined_threshold["f1"]
    ]
})

display(cnn_thresholds)

,feature_set,threshold,precision,recall,f1
0,signature,0.05,0.200000,1.0,0.333333
1,garch,0.46,0.200361,1.0,0.333835
2,combined,0.05,0.200000,1.0,0.333333


In [97]:
cnn_signature_validation_result = evaluate_classifier(
    model_name="CNN Signature — Validation",
    y_true=y_validation,
    probabilities=cnn_signature_validation_prob,
    threshold=float(best_cnn_signature_threshold["threshold"])
)

cnn_garch_validation_result = evaluate_classifier(
    model_name="CNN GARCH — Validation",
    y_true=y_validation,
    probabilities=cnn_garch_validation_prob,
    threshold=float(best_cnn_garch_threshold["threshold"])
)

cnn_combined_validation_result = evaluate_classifier(
    model_name="CNN Combined — Validation",
    y_true=y_validation,
    probabilities=cnn_combined_validation_prob,
    threshold=float(best_cnn_combined_threshold["threshold"])
)

CNN Signature — Validation
threshold: 0.0500
accuracy: 0.2000
precision: 0.2000
recall: 1.0000
f1: 0.3333
roc_auc: 0.5095

Classification report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      2220
           1       0.20      1.00      0.33       555

    accuracy                           0.20      2775
   macro avg       0.10      0.50      0.17      2775
weighted avg       0.04      0.20      0.07      2775

Confusion matrix:
[[   0 2220]
 [   0  555]]
CNN GARCH — Validation
threshold: 0.4600
accuracy: 0.2018
precision: 0.2004
recall: 1.0000
f1: 0.3338
roc_auc: 0.5055

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.00      0.00      2220
           1       0.20      1.00      0.33       555

    accuracy                           0.20      2775
   macro avg       0.60      0.50      0.17      2775
weighted avg       0.84      0.20      0.07      2775

Confusion mat

In [98]:
cnn_signature_test_prob = cnn_signature.predict(
    X_signature_test_cnn,
    verbose=0
).ravel()

cnn_garch_test_prob = cnn_garch.predict(
    X_garch_test_cnn,
    verbose=0
).ravel()

cnn_combined_test_prob = cnn_combined.predict(
    X_combined_test_cnn,
    verbose=0
).ravel()

In [99]:
cnn_signature_test_result = evaluate_classifier(
    model_name="CNN Signature — Test",
    y_true=y_test,
    probabilities=cnn_signature_test_prob,
    threshold=float(best_cnn_signature_threshold["threshold"])
)

cnn_garch_test_result = evaluate_classifier(
    model_name="CNN GARCH — Test",
    y_true=y_test,
    probabilities=cnn_garch_test_prob,
    threshold=float(best_cnn_garch_threshold["threshold"])
)

cnn_combined_test_result = evaluate_classifier(
    model_name="CNN Combined — Test",
    y_true=y_test,
    probabilities=cnn_combined_test_prob,
    threshold=float(best_cnn_combined_threshold["threshold"])
)

CNN Signature — Test
threshold: 0.0500
accuracy: 0.2000
precision: 0.2000
recall: 1.0000
f1: 0.3333
roc_auc: 0.5015

Classification report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      2240
           1       0.20      1.00      0.33       560

    accuracy                           0.20      2800
   macro avg       0.10      0.50      0.17      2800
weighted avg       0.04      0.20      0.07      2800

Confusion matrix:
[[   0 2240]
 [   0  560]]
CNN GARCH — Test
threshold: 0.4600
accuracy: 0.2004
precision: 0.2001
recall: 1.0000
f1: 0.3334
roc_auc: 0.4978

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.00      0.00      2240
           1       0.20      1.00      0.33       560

    accuracy                           0.20      2800
   macro avg       0.60      0.50      0.17      2800
weighted avg       0.84      0.20      0.07      2800

Confusion matrix:
[[   1 

In [100]:
cnn_test_results = pd.DataFrame([
    cnn_signature_test_result,
    cnn_garch_test_result,
    cnn_combined_test_result
]).sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)

display(cnn_test_results)

,model,threshold,accuracy,precision,recall,f1,roc_auc
0,CNN GARCH — Test,0.46,0.200357,0.200071,1.0,0.333433,0.497797
1,CNN Signature — Test,0.05,0.200000,0.200000,1.0,0.333333,0.501532
2,CNN Combined — Test,0.05,0.200000,0.200000,1.0,0.333333,0.489686


## Ensemble

In [101]:
def find_best_ensemble_weight(
    y_true,
    cnn_probabilities,
    xgb_probabilities
):
    """
    Tune CNN/XGBoost averaging weight and classification threshold
    using validation data.
    """

    records = []

    weights = np.arange(0.0, 1.01, 0.05)

    for cnn_weight in weights:
        ensemble_probabilities = (
            cnn_weight * cnn_probabilities
            + (1.0 - cnn_weight) * xgb_probabilities
        )

        best_threshold, _ = find_best_threshold(
            y_true,
            ensemble_probabilities
        )

        records.append({
            "cnn_weight": cnn_weight,
            "xgb_weight": 1.0 - cnn_weight,
            "threshold": best_threshold["threshold"],
            "precision": best_threshold["precision"],
            "recall": best_threshold["recall"],
            "f1": best_threshold["f1"],
            "roc_auc": roc_auc_score(
                y_true,
                ensemble_probabilities
            )
        })

    results = pd.DataFrame(records)

    best_result = results.loc[
        results["f1"].idxmax()
    ]

    return best_result, results

In [102]:
best_signature_ensemble, signature_ensemble_table = (
    find_best_ensemble_weight(
        y_true=y_validation,
        cnn_probabilities=cnn_signature_validation_prob,
        xgb_probabilities=signature_validation_prob
    )
)

best_garch_ensemble, garch_ensemble_table = (
    find_best_ensemble_weight(
        y_true=y_validation,
        cnn_probabilities=cnn_garch_validation_prob,
        xgb_probabilities=garch_validation_prob
    )
)

best_combined_ensemble, combined_ensemble_table = (
    find_best_ensemble_weight(
        y_true=y_validation,
        cnn_probabilities=cnn_combined_validation_prob,
        xgb_probabilities=combined_validation_prob
    )
)

In [103]:
ensemble_settings = pd.DataFrame([
    {
        "feature_set": "signature",
        **best_signature_ensemble.to_dict()
    },
    {
        "feature_set": "garch",
        **best_garch_ensemble.to_dict()
    },
    {
        "feature_set": "combined",
        **best_combined_ensemble.to_dict()
    }
])

display(ensemble_settings)

,feature_set,cnn_weight,xgb_weight,threshold,precision,recall,f1,roc_auc
0,signature,0.15,0.85,0.17,0.200361,1.000000,0.333835,0.506018
1,garch,0.85,0.15,0.43,0.201381,0.998198,0.335148,0.510041
2,combined,0.90,0.10,0.46,0.200733,0.987387,0.333638,0.511437


In [104]:
def evaluate_ensemble_on_test(
    model_name,
    y_test,
    cnn_test_probabilities,
    xgb_test_probabilities,
    settings
):
    cnn_weight = float(settings["cnn_weight"])
    xgb_weight = float(settings["xgb_weight"])
    threshold = float(settings["threshold"])

    ensemble_probabilities = (
        cnn_weight * cnn_test_probabilities
        + xgb_weight * xgb_test_probabilities
    )

    result = evaluate_classifier(
        model_name=model_name,
        y_true=y_test,
        probabilities=ensemble_probabilities,
        threshold=threshold
    )

    return result, ensemble_probabilities

In [105]:
signature_ensemble_test_result, signature_ensemble_test_prob = (
    evaluate_ensemble_on_test(
        model_name="Ensemble Signature — Test",
        y_test=y_test,
        cnn_test_probabilities=cnn_signature_test_prob,
        xgb_test_probabilities=signature_test_prob,
        settings=best_signature_ensemble
    )
)

garch_ensemble_test_result, garch_ensemble_test_prob = (
    evaluate_ensemble_on_test(
        model_name="Ensemble GARCH — Test",
        y_test=y_test,
        cnn_test_probabilities=cnn_garch_test_prob,
        xgb_test_probabilities=garch_test_prob,
        settings=best_garch_ensemble
    )
)

combined_ensemble_test_result, combined_ensemble_test_prob = (
    evaluate_ensemble_on_test(
        model_name="Ensemble Combined — Test",
        y_test=y_test,
        cnn_test_probabilities=cnn_combined_test_prob,
        xgb_test_probabilities=combined_test_prob,
        settings=best_combined_ensemble
    )
)

Ensemble Signature — Test
threshold: 0.1700
accuracy: 0.2146
precision: 0.1997
recall: 0.9732
f1: 0.3314
roc_auc: 0.5016

Classification report:
              precision    recall  f1-score   support

           0       0.79      0.03      0.05      2240
           1       0.20      0.97      0.33       560

    accuracy                           0.21      2800
   macro avg       0.49      0.50      0.19      2800
weighted avg       0.67      0.21      0.11      2800

Confusion matrix:
[[  56 2184]
 [  15  545]]
Ensemble GARCH — Test
threshold: 0.4300
accuracy: 0.2054
precision: 0.2002
recall: 0.9929
f1: 0.3332
roc_auc: 0.4993

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.01      0.02      2240
           1       0.20      0.99      0.33       560

    accuracy                           0.21      2800
   macro avg       0.51      0.50      0.18      2800
weighted avg       0.70      0.21      0.08      2800

Confusion matri

In [106]:
final_results = pd.DataFrame([
    xgb_signature_test_result,
    xgb_garch_test_result,
    xgb_combined_test_result,

    cnn_signature_test_result,
    cnn_garch_test_result,
    cnn_combined_test_result,

    signature_ensemble_test_result,
    garch_ensemble_test_result,
    combined_ensemble_test_result
])

final_results = final_results.sort_values(
    by=["f1", "roc_auc"],
    ascending=False
).reset_index(drop=True)

display(final_results)

,model,threshold,accuracy,precision,recall,f1,roc_auc
0,XGBoost GARCH — Test,0.19,0.216429,0.200952,0.980357,0.333536,0.498162
1,CNN GARCH — Test,0.46,0.200357,0.200071,1.000000,0.333433,0.497797
2,CNN Signature — Test,0.05,0.200000,0.200000,1.000000,0.333333,0.501532
3,CNN Combined — Test,0.05,0.200000,0.200000,1.000000,0.333333,0.489686
4,Ensemble GARCH — Test,0.43,0.205357,0.200216,0.992857,0.333233,0.499341
5,Ensemble Combined — Test,0.46,0.236786,0.201213,0.948214,0.331979,0.495813
6,XGBoost Combined — Test,0.09,0.200714,0.199283,0.992857,0.331940,0.499207
7,XGBoost Signature — Test,0.09,0.210357,0.199490,0.978571,0.331418,0.501473
8,Ensemble Signature — Test,0.17,0.214643,0.199707,0.973214,0.331408,0.501566


In [107]:
final_results.to_csv(
    MODEL_DIR / "final_model_results.csv",
    index=False
)

cnn_thresholds.to_csv(
    MODEL_DIR / "cnn_thresholds.csv",
    index=False
)

ensemble_settings.to_csv(
    MODEL_DIR / "ensemble_settings.csv",
    index=False
)

In [108]:
test_predictions = pd.DataFrame({
    "actual_label": y_test,

    "xgb_signature_prob": signature_test_prob,
    "xgb_garch_prob": garch_test_prob,
    "xgb_combined_prob": combined_test_prob,

    "cnn_signature_prob": cnn_signature_test_prob,
    "cnn_garch_prob": cnn_garch_test_prob,
    "cnn_combined_prob": cnn_combined_test_prob,

    "ensemble_signature_prob": signature_ensemble_test_prob,
    "ensemble_garch_prob": garch_ensemble_test_prob,
    "ensemble_combined_prob": combined_ensemble_test_prob
})

test_predictions.to_csv(
    MODEL_DIR / "test_probabilities.csv",
    index=False
)

In [109]:
history_signature_df = pd.DataFrame(history_signature.history)
history_garch_df = pd.DataFrame(history_garch.history)
history_combined_df = pd.DataFrame(history_combined.history)

history_signature_df.to_csv(
    MODEL_DIR / "cnn_signature_history.csv",
    index=False
)

history_garch_df.to_csv(
    MODEL_DIR / "cnn_garch_history.csv",
    index=False
)

history_combined_df.to_csv(
    MODEL_DIR / "cnn_combined_history.csv",
    index=False
)

In [110]:
importance = pd.DataFrame({
    "Feature": np.arange(len(xgb_combined.feature_importances_)),
    "Importance": xgb_combined.feature_importances_
})

importance.sort_values(
    "Importance",
    ascending=False
).to_csv(
    MODEL_DIR / "xgb_combined_importance.csv",
    index=False
)